In [26]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field

In [27]:
load_dotenv(override=True)

True

In [28]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

In [29]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description = "Sentiment of the review")

In [30]:
structured_model = model.with_structured_output(SentimentSchema)

In [31]:
prompt = 'What is the sentiment of the review - The software is too bad'
structured_model.invoke(prompt)

SentimentSchema(sentiment='negative')

In [32]:
structured_model.invoke(prompt).sentiment

'negative'

In [33]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [34]:
graph = StateGraph(ReviewState)

In [35]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

In [36]:
graph.add_node('find_sentiment', find_sentiment)

In [37]:
graph.add_edge(START, 'find_sentiment')
graph.add_edge('find_sentiment', END)

In [38]:
workflow = graph.compile()

In [39]:
input_state = {"review": "This movie is so bad"}
workflow.invoke(input_state)

{'review': 'This movie is so bad', 'sentiment': 'negative'}